# Chapter 2: Letting a Model Decide

Chapter 1 ended on day 4: one paper about neural architecture *search*,
one paper about grounding answers in documents. Our scout summarized the
wrong one, because its rules matched the word "search" and had never
heard of "grounding".

Show those same two abstracts to a person and they'd sort them in about
five seconds. Not because they have a better keyword list, but because
they read what each paper is actually about. That's the one thing our
`decide` function couldn't do, and it's exactly what a language model is
good at: reading a short piece of text and judging what it means.

This chapter uses Google's Gemini API for an optional live demo near the
end, the same free setup as Grounded AI Chapter 4: no credit card, just
a free API key from
[aistudio.google.com/apikey](https://aistudio.google.com/apikey).
Everything before that runs right here in the page.

:::{card} Chapter 2 of 11 · Foundations

**Where we left off:** Our scout runs a think, act, observe loop, but the "think" step is a list of hand-written keyword rules that match words, not topics.

**What we fix now:** Hand the "think" step to a language model: describe what you care about in a sentence, show it the papers, and get back a decision your code can act on.

**By the end, you can:**
- explain which part of the agent a model replaces, and which parts stay ordinary code
- build a prompt that asks for a structured decision, and read the reply safely before acting on it
- identify why a model that can only see what's in its prompt still gets stuck
:::

```mermaid
flowchart TB
    subgraph F["Foundations"]
        direction LR
        LOOP[Agent loop] --> MOD[Model decides] --> TOOL[Tools and autonomy]
    end
    subgraph D["Continuous Development"]
        direction LR
        SCP[Scope and data] --> V1[Build v1] --> EVL[Evals and deploy]
    end
    subgraph C["Continuous Calibration, repeats"]
        direction LR
        RUN[Real use] --> ERR[Error patterns] --> FIX[Fixes] --> MET[New metrics]
    end
    F --> D --> C

    classDef active fill:#fef3c7,stroke:#d97706,stroke-width:3px,color:#78350f
    classDef done fill:#dbeafe,stroke:#93c5fd,stroke-width:1px,color:#1e3a8a
    classDef pending fill:#f3f4f6,stroke:#e5e7eb,stroke-width:1px,color:#9ca3af
    class MOD active
    class LOOP done
    class TOOL,SCP,V1,EVL,RUN,ERR,FIX,MET pending
```

## The fix: a model does the thinking, code does the rest

Here's the key idea of this chapter, and honestly of most agents you'll
ever meet: **the model only does the "think" step.** It doesn't run
searches, it doesn't write files, it doesn't touch anything. It reads
some text and writes some text back. Everything else, acting on its
decision, checking it, stopping the loop, stays ordinary Python that
you wrote and can read.

So the change from Chapter 1 is small in code and big in effect:

1. Instead of a keyword list, describe your interest **in a sentence**,
   the way you'd explain it to a colleague.
2. Put that sentence and today's papers into a **prompt**.
3. Ask the model to reply in a **fixed format** your code can read: which
   paper to pick (or none), and one sentence saying why.
4. Your code reads that reply and **acts** on it: summarize, or stop.

```mermaid
flowchart LR
    I["Your interest,<br/>in a sentence"] --> P[Prompt]
    PAP["Today's papers"] --> P
    P --> M["Model<br/>(think)"]
    M --> R["Reply:<br/>pick + reason"]
    R --> CODE["Your code<br/>(act)"]
    CODE --> OUT["Summary,<br/>or 'nothing today'"]
```

Why a fixed format instead of just letting the model chat? Because the
next step is code, and code can't act on "I'd probably go with the
second one, it looks more relevant". It can act on
`{"pick": 1, "reason": "..."}`. The format is the handshake between the
part that reads meaning and the part that does things.

:::{tip} Try it

First, the prompt itself. The cell below rebuilds Chapter 1's papers and
`summarize` function, then assembles the exact text a model would
receive for day 4. Edit `interest` and re-run it to see the prompt
change. Notice there isn't a single keyword rule anywhere.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/velnmurugan/RAGTutorial/blob/main/agentic-ai/02-letting-a-model-decide.ipynb)

If the code below doesn't let you type into it, use the badge above to
open this same notebook in Google Colab, where you can edit and run it
freely.
:::

In [ ]:
import json
import re

def summarize(paper):
    return paper["abstract"].split(". ")[0] + "."

# The same made-up papers as Chapter 1.
day_2 = [
    {"title": "Sparse Mixture-of-Experts at Scale",
     "abstract": "We route each token to 2 of 64 experts and train a 40B parameter model. Training cost falls by 30 percent."},
    {"title": "Protein Folding with Diffusion Models",
     "abstract": "We generate protein structures with a diffusion model. Predicted structures match lab results on 71 percent of targets."},
    {"title": "RAG Without Chunking",
     "abstract": "We show that RAG systems can skip chunking by indexing whole sections with a structure-aware encoder. Answer accuracy matches chunked baselines on two QA datasets."},
]
day_3 = [
    {"title": "Weather Forecasting with Graph Networks",
     "abstract": "We forecast 10 day weather on a global graph. Error falls by 12 percent against the previous best model."},
]
day_4 = [
    {"title": "Neural Architecture Search for Tiny Devices",
     "abstract": "We search over 10,000 network designs to find models that fit on a microcontroller. The best design runs in 40 milliseconds."},
    {"title": "Grounding Answers in External Documents",
     "abstract": "We make a language model answer only from passages fetched from a document store. Unsupported claims fall by 60 percent."},
]

interest = ("papers about retrieval-augmented generation (RAG), or about "
            "making language models answer from documents they are given")

def build_prompt(papers, interest):
    listing = "\n\n".join(
        f"[{i}] {p['title']}\n{p['abstract']}" for i, p in enumerate(papers)
    )
    return f"""You help a researcher decide which new paper to read today.
Their interest: {interest}

Today's papers:
{listing}

Reply with JSON only, in exactly this shape:
{{"pick": <paper number, or null if none match the interest>, "reason": "<one sentence>"}}"""

print(build_prompt(day_4, interest))

That's everything the model will ever know about your task: one sentence
of interest, the papers, and the reply format. Keep that in mind, it
comes back at the end of this chapter.

## Plugging a model into the scout

A real model needs a network connection and an API key, and this page
can't provide either. So the next cell uses a **stand-in**: a function
called `mock_model` that returns a prewritten reply for each day. The
replies were written by hand for this chapter to show the kind of
answer a model gives; they are not recorded from a real run. The Colab
section at the end swaps the stand-in for a real Gemini model with one
line, and the rest of the code doesn't change at all.

That one-line swap is worth noticing on its own: to the scout, a model
is just a function that takes a prompt and returns text.

In [ ]:
# Hand-written stand-in replies (NOT recorded from a real model).
prewritten_replies = {
    "Sparse Mixture-of-Experts at Scale":
        '{"pick": 2, "reason": "Paper 2 is about RAG systems, which matches the interest directly."}',
    "Weather Forecasting with Graph Networks":
        '{"pick": null, "reason": "The only paper is about weather forecasting, which has nothing to do with RAG or answering from documents."}',
    "Neural Architecture Search for Tiny Devices":
        '{"pick": 1, "reason": "Paper 1 makes a model answer from fetched passages, which is grounding; paper 0 searches over network designs, not documents."}',
}

def mock_model(prompt):
    for first_title, reply in prewritten_replies.items():
        if first_title in prompt:
            return reply
    return '{"pick": null, "reason": "No prewritten reply for these papers."}'

ask_model = mock_model   # the Colab section swaps this for a real model

def decide(papers):
    reply = ask_model(build_prompt(papers, interest))   # think
    return json.loads(reply)

def run_scout(papers):
    decision = decide(papers)
    print("Model's reason:", decision["reason"])
    if decision["pick"] is None:                         # act
        print("Stop -> Nothing relevant today.")
    else:
        paper = papers[decision["pick"]]
        print(f"Summarize '{paper['title']}'")
        print("  Summary:", summarize(paper))

run_scout(day_4)

Day 4 goes the right way this time, and the reason says *why*: one
paper fetches passages to ground answers, the other searches over
network designs. That's a judgment about meaning, not spelling, and
it's exactly what the keyword rules couldn't make.

Try `run_scout(day_2)` and `run_scout(day_3)` too. On day 3 the model
says "none of these", and the scout stops honestly, just like the loop
in Chapter 1, except nobody had to write a rule for it.

## Reading the model's reply safely

There's a quiet assumption in `decide`: that the model replies with
clean JSON and a paper number that actually exists. Real models usually
do. Not always.

:::{tip} Predict before you run it

Below are two replies a model could plausibly send. The first wraps its
JSON in a friendly sentence and a code fence. The second is clean JSON,
but picks paper number 7 on a day with only two papers. What do you
expect `json.loads` to do with each one?
:::

In [ ]:
chatty_reply = 'Sure! Here is my decision:\n```json\n{"pick": 1, "reason": "Paper 1 is about grounding answers in documents."}\n```'
wrong_number_reply = '{"pick": 7, "reason": "Paper 7 is the most relevant."}'

for reply in [chatty_reply, wrong_number_reply]:
    try:
        decision = json.loads(reply)
        print("Parsed:", decision)
        print("  Paper:", day_4[decision["pick"]]["title"])
    except Exception as error:
        print("Crashed:", type(error).__name__, "-", error)

Both crash, for different reasons. The chatty reply isn't valid JSON as
a whole, even though a perfectly good decision is sitting inside it. The
second reply parses fine, but paper 7 doesn't exist, so the crash
happens one step later, when the code tries to act on it.

The lesson is bigger than JSON: **a model's reply is data, not a
command.** Your code should check it before doing anything with it, and
fall back to something safe, like stopping, when it doesn't make sense.
Here's a `parse_reply` that does both:

In [ ]:
def parse_reply(reply, papers):
    match = re.search(r"\{.*\}", reply, re.DOTALL)        # find the JSON inside the text
    if not match:
        return {"pick": None, "reason": "Could not find a decision in the model's reply."}
    try:
        decision = json.loads(match.group())
    except json.JSONDecodeError:
        return {"pick": None, "reason": "The model's reply was not valid JSON."}
    pick = decision.get("pick")
    if pick is not None and not (isinstance(pick, int) and 0 <= pick < len(papers)):
        return {"pick": None, "reason": f"The model picked paper {pick}, which doesn't exist."}
    return decision

for reply in [chatty_reply, wrong_number_reply]:
    print(parse_reply(reply, day_4))

def decide(papers):
    reply = ask_model(build_prompt(papers, interest))    # think
    return parse_reply(reply, papers)                     # check before acting

run_scout(day_4)

The chatty reply now works, because we pull the JSON out of the text.
The impossible pick turns into a safe "stop" with an honest reason,
instead of a crash. And `decide` now checks every reply before the
scout acts on it. We'll come back to this idea, checking what the model
says before trusting it, many times in this course.

:::{warning} Where it breaks

The model can only judge what we put in its prompt. It sees a title and
an abstract, and nothing else. Try a day where the abstract doesn't say
enough (same [Colab link](https://colab.research.google.com/github/velnmurugan/RAGTutorial/blob/main/agentic-ai/02-letting-a-model-decide.ipynb) as before, if you need it):
:::

:::{tip} Predict before you run it

Day 5 has a paper called "Better Answers with Less Compute". It might be
about retrieval, or it might be about something else entirely: the
abstract never says *how* the answers get better. What do you expect
the model to do, and what would *you* do in its place?
:::

In [ ]:
day_5 = [
    {"title": "Better Answers with Less Compute",
     "abstract": "We introduce a simple method that improves answer quality on two question answering benchmarks while using less compute. Code is available online."},
    {"title": "Faster Training for Speech Models",
     "abstract": "We cut speech model training time in half with a new batching scheme. Word error rate stays the same."},
]

prewritten_replies["Better Answers with Less Compute"] = (
    '{"pick": null, "reason": "Paper 0 might use retrieval, but its abstract never says '
    'how answers are improved, so I cannot tell from what I was given."}'
)

run_scout(day_5)

The model (well, our stand-in, but a careful real model often answers
the same way) says it can't tell. That's an honest answer, and a much
better one than guessing. But look at what a person would do next: open
the paper and skim the method section. The answer is right there, one
click away.

The scout can't do that. It has no way to ask for more. Everything it
knows was pasted into the prompt before it started thinking, and when
that isn't enough, its only options are "pick", "none", or guess. The
same wall shows up at scale: real arXiv days have hundreds of new
papers, and you can't paste every full text into one prompt. The model
can think now, but it can only think about what it was handed.

## Bonus (Colab only): a real model decides

Everything above runs anywhere, because `mock_model` is just a
dictionary lookup. To see a real model make these decisions, open this
notebook in Colab and run the cell below:

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/velnmurugan/RAGTutorial/blob/main/agentic-ai/02-letting-a-model-decide.ipynb)

Get a free key at [aistudio.google.com/apikey](https://aistudio.google.com/apikey)
(no credit card needed), then paste it into `API_KEY` below. Treat it
like a password: fine to paste into your own private Colab copy while
experimenting, never commit it to a public repository or notebook.

In [ ]:
# Colab only: needs a live network call and an API key.
!pip install -q -U google-genai

from google import genai

API_KEY = "YOUR_API_KEY_HERE"
client = genai.Client(api_key=API_KEY)
MODEL = "gemini-3.5-flash"

def gemini_model(prompt):
    response = client.models.generate_content(model=MODEL, contents=prompt)
    return response.text

ask_model = gemini_model   # the one-line swap

for name, papers in [("day 2", day_2), ("day 3", day_3), ("day 4", day_4), ("day 5", day_5)]:
    print(f"=== {name} ===")
    run_scout(papers)
    print()

Compare the real reasons with the stand-in ones. The wording will
differ, and on day 5 a real model might pick paper 0 with a hopeful
guess instead of saying it can't tell. Run it a few times: the answers
can change from run to run. Hold on to that observation, measuring
exactly this kind of variation is a big part of the second half of this
course.

## Quick check

Before moving on, answer for yourself (no peeking ahead):

1. Of the three moves, think, act, observe, which one did the model take over? What stayed ordinary Python, and why does that split matter?

<details>
<summary>Check your reasoning ▸</summary>

A strong answer says the model only took over "think": it reads text and writes a decision back. Acting on that decision (summarizing, stopping) and checking the reply stayed in your own code. That matters because the code is the part you can read, test, and trust; the model only ever produces text, and nothing happens unless your code chooses to act on it.

</details>

2. Chapter 1 used a keyword list. This chapter describes the interest in a sentence. Why does that fix day 4, where the keyword rules failed?

<details>
<summary>Check your reasoning ▸</summary>

A good answer explains that keywords match spelling, so "search" in "architecture search" looked relevant and "grounding" was never on the list. A sentence describes a topic, and the model judges each abstract by meaning, so it can tell that searching over network designs isn't about documents, and that fetching passages to ground answers is.

</details>

3. Why does the prompt ask for JSON in a fixed shape instead of letting the model answer freely?

<details>
<summary>Check your reasoning ▸</summary>

Because the next step is code, and code needs something it can reliably read, like a paper number or null. A free-text answer ("I'd probably go with the second one") would need a person to interpret it. The fixed format is the handshake between the part that judges meaning and the part that acts.

</details>

4. `parse_reply` turned "pick paper 7" into a safe stop. Why should code check a model's decision instead of just trusting it?

<details>
<summary>Check your reasoning ▸</summary>

Because a model's reply is text that can be malformed or simply wrong, even when it usually isn't. Acting on it blindly means one bad reply crashes the program or, worse, does the wrong thing. A strong answer adds that the safe fallback matters as much as the check: when the reply doesn't make sense, stopping honestly beats guessing.

</details>

5. On day 5 the model said it couldn't tell. What would a person do next, and what would the scout need in order to do the same thing?

<details>
<summary>Check your reasoning ▸</summary>

A person would open the paper and read more of it. The scout would need a way to *ask for* more information during its run, for example an action like "read the full text of paper 0", which its code then carries out and feeds back into the next "think". Right now the model can only judge what was pasted in before it started. A strong answer connects this back to the loop: the model needs actions it can choose between, not just a single verdict.

</details>

---

**You can now:** explain which part of an agent a model replaces, build a prompt that returns a structured decision, and check that decision before your code acts on it.

**But one problem remains:** the model can only judge what was pasted into its prompt. When an abstract doesn't say enough, it can't go and read more, and on a real day there are far too many papers to paste in at once. The next chapter gives the model actions it can ask for, like searching papers and reading one in full, and then asks an uncomfortable question: once it can act, how much should it be allowed to do without you?

**Next → Chapter 3: Tools and the Autonomy Dial**